# Submission 09

Experiment 25 synthetic identity features.

Previous leaderboard benchmark: **0.947540 ROC-AUC**

This submission uses the same feature idea as Experiment 25 and trains the final XGBoost model for the Kaggle test set.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from xgboost import XGBClassifier

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name != "DataCompetition":
    PROJECT_ROOT = Path.home() / "Documents" / "DataCompetition"

DATA_DIR = PROJECT_ROOT / "data"

train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")

TARGET = "Will_Buy_EV"
ID_COL = "id"

RANDOM_STATE = 42
N_SPLITS = 3
SMOOTHING = 20.0

NUM_COLS = [
    "Age",
    "Annual_Income_USD",
    "Daily_Commute_km",
    "Number_of_Cars_Owned",
    "Charging_Stations_Near_Home",
    "Charging_Stations_Near_Work",
    "Environmental_Concern_Level",
]

CAT_COLS = [
    "Gender",
    "City_Type",
    "Current_Car_Type",
    "Home_Charging_Possible",
    "Subsidy_Available",
    "Range_Anxiety_Level",
]

X_train_raw = train.drop(columns=[TARGET, ID_COL])
X_test_raw = test.drop(columns=[ID_COL])

y = train[TARGET].map({"No": 0, "Yes": 1}).astype(np.int8)

X_all = pd.concat(
    [X_train_raw, X_test_raw],
    axis=0,
    ignore_index=True,
)

X_all = pd.get_dummies(
    X_all,
    columns=CAT_COLS,
    dtype=np.int8,
)

X_base = X_all.iloc[:len(train)].copy()
X_test_base = X_all.iloc[len(train):].copy()

print(f"Train shape: {train.shape}")
print(f"Test shape: {test.shape}")
print(f"Base feature count: {X_base.shape[1]}")


In [ ]:
IDENTITY_COLS = [
    "Age",
    "Annual_Income_USD",
    "Daily_Commute_km",
    "Number_of_Cars_Owned",
    "Charging_Stations_Near_Home",
    "Charging_Stations_Near_Work",
    "Environmental_Concern_Level",
]

IDENTITY_PAIRS = [
    ("Age", "Annual_Income_USD"),
    ("Age", "Daily_Commute_km"),
    ("Age", "Current_Car_Type"),
    ("Annual_Income_USD", "Current_Car_Type"),
    ("Annual_Income_USD", "City_Type"),
    ("Daily_Commute_km", "Current_Car_Type"),
    ("Charging_Stations_Near_Home", "Charging_Stations_Near_Work"),
    ("Environmental_Concern_Level", "Range_Anxiety_Level"),
]


In [ ]:
def add_identity_features(
    train_X,
    test_X,
    raw_train,
    raw_test,
    y,
    columns,
):
    train_out = train_X.copy()
    test_out = test_X.copy()

    skf = StratifiedKFold(
        n_splits=N_SPLITS,
        shuffle=True,
        random_state=RANDOM_STATE,
    )

    global_mean = y.mean()

    for col in columns:
        train_values = raw_train[col]
        test_values = raw_test[col]

        train_te = np.zeros(len(train_X))
        train_freq = np.zeros(len(train_X))

        for tr_idx, va_idx in skf.split(raw_train, y):
            stats = pd.DataFrame({
                "key": train_values.iloc[tr_idx].to_numpy(),
                "target": y.iloc[tr_idx].to_numpy(),
            })

            grouped = (
                stats.groupby("key", dropna=False)["target"]
                .agg(["sum", "count"])
            )

            sums = (
                train_values.iloc[va_idx]
                .map(grouped["sum"])
                .fillna(0)
                .to_numpy()
            )

            counts = (
                train_values.iloc[va_idx]
                .map(grouped["count"])
                .fillna(0)
                .to_numpy()
            )

            train_te[va_idx] = (
                sums + SMOOTHING * global_mean
            ) / (
                counts + SMOOTHING
            )

            train_freq[va_idx] = counts / len(tr_idx)

        full_stats = pd.DataFrame({
            "key": train_values.to_numpy(),
            "target": y.to_numpy(),
        })

        full_grouped = (
            full_stats.groupby("key", dropna=False)["target"]
            .agg(["sum", "count"])
        )

        test_sums = (
            test_values
            .map(full_grouped["sum"])
            .fillna(0)
            .to_numpy()
        )

        test_counts = (
            test_values
            .map(full_grouped["count"])
            .fillna(0)
            .to_numpy()
        )

        test_te = (
            test_sums + SMOOTHING * global_mean
        ) / (
            test_counts + SMOOTHING
        )

        test_freq = test_counts / len(train)

        train_out[f"{col}__te"] = train_te
        train_out[f"{col}__freq"] = train_freq

        test_out[f"{col}__te"] = test_te
        test_out[f"{col}__freq"] = test_freq

    return train_out, test_out


In [ ]:
def add_pair_features(
    train_X,
    test_X,
    raw_train,
    raw_test,
    y,
    pairs,
):
    train_out = train_X.copy()
    test_out = test_X.copy()

    skf = StratifiedKFold(
        n_splits=N_SPLITS,
        shuffle=True,
        random_state=RANDOM_STATE,
    )

    global_mean = y.mean()

    for cols in pairs:
        name = "__".join(cols)

        train_keys = (
            raw_train[list(cols)]
            .astype(str)
            .agg("||".join, axis=1)
        )

        test_keys = (
            raw_test[list(cols)]
            .astype(str)
            .agg("||".join, axis=1)
        )

        train_te = np.zeros(len(train_X))
        train_freq = np.zeros(len(train_X))

        for tr_idx, va_idx in skf.split(raw_train, y):
            stats = pd.DataFrame({
                "key": train_keys.iloc[tr_idx].to_numpy(),
                "target": y.iloc[tr_idx].to_numpy(),
            })

            grouped = (
                stats.groupby("key", dropna=False)["target"]
                .agg(["sum", "count"])
            )

            sums = (
                train_keys.iloc[va_idx]
                .map(grouped["sum"])
                .fillna(0)
                .to_numpy()
            )

            counts = (
                train_keys.iloc[va_idx]
                .map(grouped["count"])
                .fillna(0)
                .to_numpy()
            )

            train_te[va_idx] = (
                sums + SMOOTHING * global_mean
            ) / (
                counts + SMOOTHING
            )

            train_freq[va_idx] = counts / len(tr_idx)

        full_stats = pd.DataFrame({
            "key": train_keys.to_numpy(),
            "target": y.to_numpy(),
        })

        full_grouped = (
            full_stats.groupby("key", dropna=False)["target"]
            .agg(["sum", "count"])
        )

        test_sums = (
            test_keys
            .map(full_grouped["sum"])
            .fillna(0)
            .to_numpy()
        )

        test_counts = (
            test_keys
            .map(full_grouped["count"])
            .fillna(0)
            .to_numpy()
        )

        test_te = (
            test_sums + SMOOTHING * global_mean
        ) / (
            test_counts + SMOOTHING
        )

        test_freq = test_counts / len(train)

        train_out[f"{name}__te"] = train_te
        train_out[f"{name}__freq"] = train_freq

        test_out[f"{name}__te"] = test_te
        test_out[f"{name}__freq"] = test_freq

    return train_out, test_out


In [ ]:
X_identity, X_test_identity = add_identity_features(
    X_base,
    X_test_base,
    X_train_raw,
    X_test_raw,
    y,
    IDENTITY_COLS,
)

X_identity, X_test_identity = add_pair_features(
    X_identity,
    X_test_identity,
    X_train_raw,
    X_test_raw,
    y,
    IDENTITY_PAIRS,
)

print(f"Final train features: {X_identity.shape[1]}")
print(f"Final test features: {X_test_identity.shape[1]}")


In [ ]:
XGB_PARAMS = dict(
    n_estimators=1000,
    max_depth=5,
    learning_rate=0.035,
    min_child_weight=2,
    subsample=0.90,
    colsample_bytree=0.85,
    gamma=0,
    reg_alpha=0,
    reg_lambda=1,
    objective="binary:logistic",
    eval_metric="auc",
    tree_method="hist",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

model = XGBClassifier(**XGB_PARAMS)

model.fit(
    X_identity,
    y,
)

test_predictions = model.predict_proba(
    X_test_identity
)[:, 1]

submission = pd.DataFrame({
    ID_COL: test[ID_COL],
    TARGET: test_predictions,
})

submission_path = DATA_DIR / "submission_09.csv"

submission.to_csv(
    submission_path,
    index=False,
)

print(f"Saved: {submission_path}")
print(submission.head())
print(f"Prediction range: {test_predictions.min():.6f} to {test_predictions.max():.6f}")
